# Civil Law → Neo4j Aura  
Loads **1093 articles** from the Egyptian Civil Law JSON, generates multilingual embeddings with **BAAI/bge-m3**, and inserts everything into Neo4j with intelligent keyword-based and semantic relationships.

**Graph schema**
```
(Article) -[:HAS_KEYWORD]->    (Keyword)
(Article) -[:IN_SECTION]->     (Section)
(Section) -[:HAS_SUBSECTION]-> (Section)
(Article) -[:SIMILAR_TO {score}]-> (Article)   ← semantic similarity
(Article) -[:SAME_TOPIC]->     (Article)        ← shared keywords
```

### Steps
1. Install dependencies  
2. Upload the JSON file  
3. Configure Neo4j credentials  
4. Generate embeddings (batched, GPU-accelerated)  
5. Insert nodes + relationships  
6. Create vector index for semantic search  
7. Verify & sample queries  

## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install neo4j FlagEmbedding transformers torch tqdm

## Step 2 — Upload the JSON file
Run the cell below and upload `1576751803.json` when prompted.

In [ ]:
from google.colab import files
import json, os

uploaded = files.upload()          # select 1576751803.json
JSON_PATH = list(uploaded.keys())[0]
print(f'Loaded: {JSON_PATH}')

with open(JSON_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

articles = {k: v for k, v in raw.items() if k.startswith('Article')}
print(f'Articles found: {len(articles)}')

## Step 3 — Neo4j Credentials

In [ ]:
NEO4J_URI      = 'neo4j+s://785ea338.databases.neo4j.io'
NEO4J_USER     = '785ea338'
NEO4J_PASSWORD = '2e3Vah_a8qA5Q14DaECSa87pj0LifbWK_sJq6kZWbsE'
NEO4J_DATABASE = '785ea338'

from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print('Connected to Neo4j Aura!')

## Step 4 — Schema Setup (Constraints + Indexes)

In [ ]:
SCHEMA_QUERIES = [
    'CREATE CONSTRAINT article_id IF NOT EXISTS FOR (a:Article) REQUIRE a.id IS UNIQUE',
    'CREATE CONSTRAINT keyword_name IF NOT EXISTS FOR (k:Keyword) REQUIRE k.name IS UNIQUE',
    'CREATE CONSTRAINT section_name IF NOT EXISTS FOR (s:Section) REQUIRE s.name IS UNIQUE',
]

with driver.session(database=NEO4J_DATABASE) as session:
    for q in SCHEMA_QUERIES:
        session.run(q)
print('Schema ready.')

## Step 5 — Load BGE-M3 Embedding Model  
BGE-M3 is multilingual and handles Arabic + English in a single pass.

In [ ]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
print('BGE-M3 loaded.')

## Step 6 — Parse Articles and Generate Embeddings
We embed the concatenation of the English text + Arabic text so the vector captures both languages.

In [ ]:
from tqdm.auto import tqdm
import numpy as np

BATCH_SIZE = 16   # reduce to 8 if you hit GPU OOM

records = []
for art_id, body in articles.items():
    records.append({
        'id':       art_id,
        'number':   int(art_id.replace('Article ', '')),
        'arabic':   body.get('arabic', '').strip(),
        'english':  body.get('english', '').strip(),
        'metadata': body.get('metadata', []),
    })

# Build texts for embedding: english + arabic (BGE-M3 handles mixed lang)
texts = [f"{r['english']}\n{r['arabic']}" for r in records]

print(f'Generating embeddings for {len(texts)} articles...')
all_embeddings = []
for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i : i + BATCH_SIZE]
    out = model.encode(batch, batch_size=BATCH_SIZE, max_length=512)
    all_embeddings.extend(out['dense_vecs'].tolist())

for i, rec in enumerate(records):
    rec['embedding'] = all_embeddings[i]

EMBED_DIM = len(records[0]['embedding'])
print(f'Done. Embedding dimension: {EMBED_DIM}')

## Step 7 — Insert Article Nodes

In [ ]:
INSERT_ARTICLE_Q = """
UNWIND $rows AS row
MERGE (a:Article {id: row.id})
SET   a.number    = row.number,
      a.arabic    = row.arabic,
      a.english   = row.english,
      a.embedding = row.embedding
"""

CHUNK = 50
with driver.session(database=NEO4J_DATABASE) as session:
    for i in tqdm(range(0, len(records), CHUNK), desc='Inserting articles'):
        batch = records[i : i + CHUNK]
        session.run(INSERT_ARTICLE_Q, rows=[
            {'id': r['id'], 'number': r['number'],
             'arabic': r['arabic'], 'english': r['english'],
             'embedding': r['embedding']}
            for r in batch
        ])

print('Articles inserted.')

## Step 8 — Insert Keywords and Sections, Link to Articles

The `metadata` list for each article is a **hierarchical path** of section headings (Arabic + English, alternating).  
We create a `Section` hierarchy and tag each article with its leaf `Keyword` nodes.

In [ ]:
def detect_lang(text):
    """Simple heuristic: if >30% Arabic unicode chars → Arabic."""
    ar = sum(1 for c in text if '\u0600' <= c <= '\u06ff')
    return 'ar' if ar / max(len(text), 1) > 0.3 else 'en'

# ── 8a: Keyword nodes + HAS_KEYWORD relationships ─────────────────────────
KEYWORD_Q = """
UNWIND $rows AS row
MERGE (k:Keyword {name: row.keyword})
SET   k.lang = row.lang
WITH  k, row
MATCH (a:Article {id: row.article_id})
MERGE (a)-[:HAS_KEYWORD]->(k)
"""

kw_rows = []
for r in records:
    for kw in r['metadata']:
        kw_clean = kw.strip()
        if kw_clean:
            kw_rows.append({
                'article_id': r['id'],
                'keyword':    kw_clean,
                'lang':       detect_lang(kw_clean)
            })

with driver.session(database=NEO4J_DATABASE) as session:
    for i in tqdm(range(0, len(kw_rows), 200), desc='Keywords'):
        session.run(KEYWORD_Q, rows=kw_rows[i : i + 200])

print(f'Keywords done ({len(kw_rows)} relationships).')

# ── 8b: Section hierarchy ─────────────────────────────────────────────────
# We treat each unique metadata path as a chain: root → … → leaf
# and link (parent Section)-[:HAS_SUBSECTION]->(child Section)
SECTION_Q = """
UNWIND $pairs AS pair
MERGE (parent:Section {name: pair.parent})
MERGE (child:Section  {name: pair.child})
MERGE (parent)-[:HAS_SUBSECTION]->(child)
"""

ARTICLE_SECTION_Q = """
UNWIND $rows AS row
MATCH (a:Article  {id:   row.article_id})
MERGE (s:Section  {name: row.section})
MERGE (a)-[:IN_SECTION]->(s)
"""

seen_pairs = set()
sec_pairs  = []
art_sec    = []

for r in records:
    path = [m.strip() for m in r['metadata'] if m.strip()]
    if not path:
        continue
    # Link each consecutive pair in the path
    for i in range(len(path) - 1):
        pair = (path[i], path[i+1])
        if pair not in seen_pairs:
            seen_pairs.add(pair)
            sec_pairs.append({'parent': path[i], 'child': path[i+1]})
    # Article belongs to the leaf section
    art_sec.append({'article_id': r['id'], 'section': path[-1]})

with driver.session(database=NEO4J_DATABASE) as session:
    for i in tqdm(range(0, len(sec_pairs), 200), desc='Sections'):
        session.run(SECTION_Q, pairs=sec_pairs[i : i + 200])
    for i in tqdm(range(0, len(art_sec), 200), desc='Article→Section'):
        session.run(ARTICLE_SECTION_Q, rows=art_sec[i : i + 200])

print('Sections done.')

## Step 9 — SAME_TOPIC Relationships (shared keywords)
Articles sharing **2 or more** keywords are connected with `SAME_TOPIC {shared_count}`.

In [ ]:
SAME_TOPIC_Q = """
MATCH (a:Article)-[:HAS_KEYWORD]->(k:Keyword)<-[:HAS_KEYWORD]-(b:Article)
WHERE a.number < b.number
WITH  a, b, count(k) AS shared
WHERE shared >= 2
MERGE (a)-[r:SAME_TOPIC]-(b)
SET   r.shared_keywords = shared
"""

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run(SAME_TOPIC_Q)
    summary = result.consume()
    print(f'SAME_TOPIC relationships created: {summary.counters.relationships_created}')

## Step 10 — Vector Index for Semantic Search

In [ ]:
VECTOR_INDEX_Q = f"""
CREATE VECTOR INDEX article_embedding IF NOT EXISTS
FOR (a:Article) ON (a.embedding)
OPTIONS {{
  indexConfig: {{
    `vector.dimensions`:       {EMBED_DIM},
    `vector.similarity_function`: 'cosine'
  }}
}}
"""

with driver.session(database=NEO4J_DATABASE) as session:
    session.run(VECTOR_INDEX_Q)

print(f'Vector index created (dim={EMBED_DIM}, cosine).')

## Step 11 — Semantic SIMILAR_TO Relationships
For every article we find the top-5 most similar articles via the vector index and create `SIMILAR_TO {score}` edges.  
Only links with **cosine score ≥ 0.85** and **different leaf sections** are kept (avoids trivial same-article links).

In [ ]:
SEMANTIC_SEARCH_Q = """
CALL db.index.vector.queryNodes('article_embedding', $topK, $embedding)
YIELD node AS candidate, score
WHERE candidate.id <> $article_id AND score >= $threshold
RETURN candidate.id AS cid, score
"""

SIMILAR_TO_Q = """
MATCH (a:Article {id: $aid})
MATCH (b:Article {id: $bid})
MERGE (a)-[r:SIMILAR_TO]-(b)
SET   r.score = $score
"""

TOP_K     = 6    # fetch 6 because one result will be the article itself
THRESHOLD = 0.85

total_sim = 0
with driver.session(database=NEO4J_DATABASE) as session:
    for rec in tqdm(records, desc='Semantic links'):
        results = session.run(
            SEMANTIC_SEARCH_Q,
            topK=TOP_K,
            embedding=rec['embedding'],
            article_id=rec['id'],
            threshold=THRESHOLD
        )
        for row in results:
            session.run(SIMILAR_TO_Q,
                        aid=rec['id'], bid=row['cid'],
                        score=round(row['score'], 4))
            total_sim += 1

print(f'SIMILAR_TO relationships created: {total_sim}')

## Step 12 — Verify: Graph Statistics

In [ ]:
STATS_Q = """
MATCH (n) RETURN labels(n)[0] AS label, count(n) AS count
UNION ALL
MATCH ()-[r]->() RETURN type(r) AS label, count(r) AS count
"""

with driver.session(database=NEO4J_DATABASE) as session:
    rows = session.run(STATS_Q).data()

print(f'{"Type":<25} {"Count":>8}')
print('-' * 35)
for row in rows:
    print(f"{row['label']:<25} {row['count']:>8}")

## Step 13 — Sample Queries

### 13a — Semantic search: find articles similar to a query

In [ ]:
QUERY_TEXT = "prescription and limitation periods"

q_vec = model.encode([QUERY_TEXT])['dense_vecs'][0].tolist()

SEMANTIC_Q = """
CALL db.index.vector.queryNodes('article_embedding', 5, $vec)
YIELD node, score
RETURN node.id AS article, score,
       LEFT(node.english, 120) AS preview
"""

with driver.session(database=NEO4J_DATABASE) as session:
    rows = session.run(SEMANTIC_Q, vec=q_vec).data()

print(f'Top results for: "{QUERY_TEXT}"\n')
for r in rows:
    print(f"  {r['article']:12}  score={r['score']:.4f}  →  {r['preview']}...")

### 13b — Keyword graph: which articles share a keyword?

In [ ]:
KEYWORD_SEARCH = "prescription"

KW_Q = """
MATCH (a:Article)-[:HAS_KEYWORD]->(k:Keyword)
WHERE toLower(k.name) CONTAINS toLower($kw)
RETURN a.id AS article, k.name AS keyword
ORDER BY a.number
LIMIT 20
"""

with driver.session(database=NEO4J_DATABASE) as session:
    rows = session.run(KW_Q, kw=KEYWORD_SEARCH).data()

for r in rows:
    print(f"  {r['article']:12}  keyword: {r['keyword']}")

### 13c — Section tree: browse the law hierarchy

In [ ]:
TREE_Q = """
MATCH path = (root:Section)-[:HAS_SUBSECTION*1..3]->(leaf:Section)
WHERE NOT ()-[:HAS_SUBSECTION]->(root)
RETURN [n IN nodes(path) | n.name] AS hierarchy
LIMIT 20
"""

with driver.session(database=NEO4J_DATABASE) as session:
    rows = session.run(TREE_Q).data()

for r in rows:
    print(' → '.join(r['hierarchy']))

### 13d — Related articles: neighborhood of Article 1

In [ ]:
NEIGHBOR_Q = """
MATCH (a:Article {id: 'Article 1'})-[r]-(b)
RETURN type(r) AS rel, labels(b)[0] AS type,
       CASE WHEN b:Article THEN b.id ELSE b.name END AS target,
       CASE WHEN r.score IS NOT NULL THEN r.score
            WHEN r.shared_keywords IS NOT NULL THEN r.shared_keywords
            ELSE null END AS weight
ORDER BY rel, target
"""

with driver.session(database=NEO4J_DATABASE) as session:
    rows = session.run(NEIGHBOR_Q).data()

for r in rows:
    w = f"  weight={r['weight']}" if r['weight'] else ''
    print(f"  [{r['rel']:15}] → {r['type']:10} {r['target']}{w}")

---
## Done
Your Neo4j graph now contains:
- **Article** nodes with Arabic, English text, and a 1024-dim BGE-M3 embedding
- **Keyword** nodes linked to articles via `HAS_KEYWORD`
- **Section** nodes forming a hierarchy via `HAS_SUBSECTION`; articles linked via `IN_SECTION`
- `SAME_TOPIC` edges between articles sharing ≥ 2 keywords
- `SIMILAR_TO` semantic edges (cosine ≥ 0.85) powered by the vector index

Open **Neo4j Aura → Explore** to browse the graph visually.